In [1]:
import os, sys
sys.path.append(os.path.abspath(os.path.join(os.curdir, '..')))
from src.utils import *
from src.panda_program import PandaMugProgram
from pydrake.all import (
    StartMeshcat,
    Quaternion,
    RigidTransform,
    RotationMatrix,
    MinimumDistanceLowerBoundConstraint,
    SceneGraphInspector,
)



ikflow/config.py | Using device: 'cuda:0'


In [2]:
yaml_file = os.path.join(RepoDir(), "models/panda/panda_finray_collision.yaml")
meshcat = StartMeshcat()
diagram = BuildEnv(meshcat=meshcat, directives_file = yaml_file)
program = PandaMugProgram(diagram)


program.plant.num_positions()
target_mug = Mug(RigidTransform(), height = 0.3)
program.create_prog(target_mug=target_mug)



INFO:drake:Meshcat listening for connections at http://localhost:7000


WorldModel::LoadRobot: /home/tangles/.cache/jrl/urdfs/panda_arm_hand_formatted_link_filepaths_absolute.urdf
joint mimic: no multiplier, using default value of 1 
joint mimic: no offset, using default value of 0 
URDFParser: Link size: 17
URDFParser: Joint size: 12
Geometry: Loading 12 meshes from /home/tangles/Urop/ikflow/.venv/lib/python3.10/site-packages/jrl/urdfs/panda/meshes/visual/link0.dae into Group
Geometry: Loading 4 meshes from /home/tangles/Urop/ikflow/.venv/lib/python3.10/site-packages/jrl/urdfs/panda/meshes/visual/link3.dae into Group
Geometry: Loading 4 meshes from /home/tangles/Urop/ikflow/.venv/lib/python3.10/site-packages/jrl/urdfs/panda/meshes/visual/link4.dae into Group
Geometry: Loading 3 meshes from /home/tangles/Urop/ikflow/.venv/lib/python3.10/site-packages/jrl/urdfs/panda/meshes/visual/link5.dae into Group
Geometry: Loading 17 meshes from /home/tangles/Urop/ikflow/.venv/lib/python3.10/site-packages/jrl/urdfs/panda/meshes/visual/link6.dae into Group
Geometry: Loa

In [3]:
while True:
    q = np.random.uniform(low=program.plant.GetPositionLowerLimits(), high=program.plant.GetPositionUpperLimits())
    if program.collision_free_constraint.Eval(q) < 1:
        break

def GenerateDiagramWithMug(q):
    program.SetPositions(q)
    target = program.frame.CalcPoseInWorld(program.plant_context)
    translation = target.translation()
    rotation = target.rotation().ToRollPitchYaw().vector() * 180 / np.pi

    mug_str = f"""\n  - add_model:\n      name: mug\n      # file: package://drake/examples/manipulation_station/models/shelves.sdf\n      file: package://combining_kinematics/models/mug/mug_simple_red.urdf\n  - add_weld:\n      parent: world\n      child: mug::mug_body_link\n      X_PC:\n        translation: [{translation[0]}, {translation[1]}, {translation[2]}]\n        rotation: !Rpy {{ deg: [{rotation[0]}, {rotation[1]}, {rotation[2]}] }}
    """
    original_size = os.path.getsize(yaml_file)
    with open(yaml_file, "a+") as f:
        f.write(mug_str)
    meshcat_with_mug = StartMeshcat()
    diagram_with_mug = BuildEnv(meshcat=meshcat_with_mug, directives_file = yaml_file)
    with open(yaml_file, "r+") as f:
        f.truncate(original_size)
    return diagram_with_mug, meshcat_with_mug, Mug(target)




In [4]:
diagram_with_mug, meshcat_with_mug, mug = GenerateDiagramWithMug(q)
program = PandaMugProgram(diagram_with_mug)
program.SetPositions(q)
program.diagram.ForcedPublish(program.diagram_context)

INFO:drake:Meshcat listening for connections at http://localhost:7001


WorldModel::LoadRobot: /home/tangles/.cache/jrl/urdfs/panda_arm_hand_formatted_link_filepaths_absolute.urdf
joint mimic: no multiplier, using default value of 1 
joint mimic: no offset, using default value of 0 
URDFParser: Link size: 17
URDFParser: Joint size: 12
Geometry: Loading 12 meshes from /home/tangles/Urop/ikflow/.venv/lib/python3.10/site-packages/jrl/urdfs/panda/meshes/visual/link0.dae into Group
Geometry: Loading 4 meshes from /home/tangles/Urop/ikflow/.venv/lib/python3.10/site-packages/jrl/urdfs/panda/meshes/visual/link3.dae into Group
Geometry: Loading 4 meshes from /home/tangles/Urop/ikflow/.venv/lib/python3.10/site-packages/jrl/urdfs/panda/meshes/visual/link4.dae into Group
Geometry: Loading 3 meshes from /home/tangles/Urop/ikflow/.venv/lib/python3.10/site-packages/jrl/urdfs/panda/meshes/visual/link5.dae into Group
Geometry: Loading 17 meshes from /home/tangles/Urop/ikflow/.venv/lib/python3.10/site-packages/jrl/urdfs/panda/meshes/visual/link6.dae into Group
Geometry: Loa

In [ ]:
program.create_prog(target_mug=mug)
print(program.collision_free_constraint.Eval(q))
result = program.Solve()
result.is_success()

[0.84418389]


In [ ]:
q_sol = program.VarsToQ(result.get_x_val())
program.SetPositions(q_sol)
program.diagram.ForcedPublish(program.diagram_context)